# DB Indexer with skillsets

- DB source indexer
- Skills:
    - No chunk
    - TextTranslationSkill (content)
    - 2 x AzureOpenAIEmbeddingSkill (title, content)


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from azure.core.credentials import AzureKeyCredential  
from azure.search.documents import SearchClient  
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (  
    SearchIndex,  
    SearchField,  
    SearchFieldDataType,  
    SimpleField,  
    SearchableField,  
    SearchIndex,  
    SemanticConfiguration,  
    SearchField,  
    VectorSearch,
    SemanticSearch,
    SemanticPrioritizedFields,
    SemanticField,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    AzureOpenAIEmbeddingSkill,
    FieldMapping,
    IndexProjectionMode,
    InputFieldMappingEntry,
    OutputFieldMappingEntry,
    SearchIndexer,
    SearchIndexerDataContainer,
    SearchIndexerDataSourceConnection,
    SearchIndexerDataSourceType,
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    SearchIndexerSkillset,
    SplitSkill,
    TextTranslationSkill,
    CognitiveServicesAccountKey,
    VectorSearchCompression,
    BinaryQuantizationCompression
)

In [3]:
import os

search_endpoint = os.getenv("AZSCH_ENDPOINT")  
credential = AzureKeyCredential(os.environ["AZSCH_KEY"])
#print(search_endpoint)

api_key = os.environ["AZURE_OPENAI_KEY"]
azure_endpoint = os.environ['AZURE_OPENAI_ENDPOINT']

sql_servername= os.environ["SQL_SERVERNAME"]
sql_password= os.environ["SQL_PASSWORD"]
sql_table_name = os.environ["SQL_TABLE_NAME"]
sql_connection_string = f"Server=tcp:{sql_servername}.database.windows.net,1433;Initial Catalog=testsqldb;Persist Security Info=False;User ID=iljoong;Password={sql_password};MultipleActiveResultSets=False;Encrypt=True;TrustServerCertificate=False;Connection Timeout=30;"
#print(sql_servername)

# for billing ai skill
cognitive_api_key=os.environ["AZURE_COGNITIVE_KEY"]

In [4]:
# index name
index_name = 'nc-articles-nochunk-index'

In [5]:
indexer_client = SearchIndexerClient(search_endpoint, credential)

data_source_name = f"{index_name}-datasource"
data_sources = indexer_client.get_data_source_connection_names()

if (data_source_name in data_sources):
    print(f"Data source connection {data_source_name} exists. Deleting it...")
    dsc = indexer_client.get_data_source_connection(name=data_source_name)
    indexer_client.delete_data_source_connection(dsc)

data_source_connections = indexer_client.get_data_source_connections()
indexer_client.create_data_source_connection(
    data_source_connection=SearchIndexerDataSourceConnection(
        name=data_source_name, 
        type=SearchIndexerDataSourceType.AZURE_SQL,
        connection_string=sql_connection_string,
        container=SearchIndexerDataContainer(name=sql_table_name)))

## Index

In [ ]:
# Create a search index
index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)
fields = [    
    SearchableField(name="article_id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="title", type=SearchFieldDataType.String,
                    searchable=True, retrievable=True), # |analyzer_name="standard.lucene"
    SearchableField(name="content", type=SearchFieldDataType.String,
                    searchable=True, retrievable=True), # analyzer_name="standard.lucene"
    SearchableField(name="game_code", type=SearchFieldDataType.String,
                    searchable=False, retrievable=True,
                    filterable=True, facetable=True),
    SearchField(name="board_id", type=SearchFieldDataType.Int32, # for int32/int64 don't use `SearchableField``
                    filterable=True, sortable=True, facetable=True),
    SearchableField(name="updated_at", type=SearchFieldDataType.DateTimeOffset,
                    searchable=False, retrievable=True,
                    sortable=True, filterable=True),
    SearchableField(name="translated", type=SearchFieldDataType.String,
                    searchable=True, retrievable=True, 
                    analyzer_name="ko.microsoft"),
    SearchField(name="title_vector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, stored=True,
                vector_search_dimensions=3072, vector_search_profile_name="myHnswProfile"),
    SearchField(name="content_vector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, stored=True,
                vector_search_dimensions=3072, vector_search_profile_name="myHnswProfile")  
]

# indexer
vector_search = VectorSearch(  
    algorithms=[  
        HnswAlgorithmConfiguration(  
            name="myHnsw",  
            parameters=HnswParameters(  
                m=4,  
                ef_construction=400,  
                ef_search=500,  
                metric=VectorSearchAlgorithmMetric.COSINE,  
            ),  
        )
    ],  
    profiles=[  
        VectorSearchProfile(  
            name="myHnswProfile",  
            algorithm_configuration_name="myHnsw",
            vectorizer_name="vectorizer",
            #compression_name="myCompression"
        )
    ],
    vectorizers=[
      AzureOpenAIVectorizer(
        vectorizer_name="vectorizer",
        kind="azureOpenAI",
        parameters = AzureOpenAIVectorizerParameters(
            resource_url=azure_endpoint,
            api_key=api_key,
            deployment_name="text-embedding-3-large",
            model_name="text-embedding-3-large",
        ),
      )
    ]
    #compressions=[
    #    BinaryQuantizationCompression(compression_name="myCompression", truncation_dimension=1024)
    #]
)  

semantic_config = SemanticConfiguration(  
    name="semantic-config",  
    prioritized_fields=SemanticPrioritizedFields(  
        title_field=SemanticField(field_name="title"),
        content_fields=[SemanticField(field_name="content")]  
    ),  
)

semantic_ko_config = SemanticConfiguration(  
    name="semantic-ko-config",  
    prioritized_fields=SemanticPrioritizedFields(  
        title_field=SemanticField(field_name="title"),
        content_fields=[SemanticField(field_name="translated")]  
    ),  
)

# Create the semantic search with the configuration  
semantic_search = SemanticSearch(configurations=[semantic_config, semantic_ko_config]) 

# Create the search index
index = SearchIndex(name=index_name, fields=fields,
                    vector_search=vector_search, semantic_search=semantic_search)

# Check if the index already exists
try:
    if index_client.get_index(name=index_name):
        print(f"Index {index_name} already exists. Deleting it...")
        index_client.delete_index(index_client.get_index(index_name))
except Exception as e:
    if "NotFound" in str(e):
        print(f"Index {index_name} does not exist. Proceeding to create it...")

result = index_client.create_or_update_index(index)
print(f' {result.name} created')

 nc-articles-nochunk-index created


## Indexer

In [7]:
skillset_name = f"{index_name}-skillset"
skillsets = indexer_client.get_skillset_names()
if (skillset_name in skillsets):
    print(f"Skillset {skillset_name} exists. Deleting it...")
    skillset = indexer_client.get_skillset(name=skillset_name)
    indexer_client.delete_skillset(skillset)

print(f"Creating skillset: {index_name}")
indexer_client.create_skillset(
    skillset=SearchIndexerSkillset(
        name=f"{index_name}-skillset",
        skills=[
            TextTranslationSkill(
                context="/document",
                default_to_language_code="ko",
                inputs=[InputFieldMappingEntry(name="text", source="/document/content")],
                outputs=[OutputFieldMappingEntry(name="translatedText", target_name="translatedText")]),
            AzureOpenAIEmbeddingSkill(
                context="/document",
                resource_url=azure_endpoint,
                api_key=api_key,
                deployment_name="text-embedding-3-large",
                model_name="text-embedding-3-large",
                dimensions=3072,
                inputs=[InputFieldMappingEntry(name="text", source="/document/content")],
                outputs=[OutputFieldMappingEntry(name="embedding", target_name="content_vector")]),
            AzureOpenAIEmbeddingSkill(
                context="/document",
                resource_url=azure_endpoint,
                api_key=api_key,
                deployment_name="text-embedding-3-large",
                model_name="text-embedding-3-large",
                dimensions=3072,
                inputs=[InputFieldMappingEntry(name="text", source="/document/title")],
                outputs=[OutputFieldMappingEntry(name="embedding", target_name="title_vector")])
        ],
        cognitive_services_account=CognitiveServicesAccountKey(
            key=cognitive_api_key
        ),
    ))

indexers = indexer_client.get_indexer_names()
if (index_name in indexers):
    print(f"Indexer {index_name} exists. Deleting it...")
    indexer = indexer_client.get_indexer(name=index_name)
    indexer_client.delete_indexer(indexer)
    
indexer_client.create_indexer(
    indexer=SearchIndexer(
        name=index_name,
        data_source_name=data_source_name,
        skillset_name=skillset_name,
        target_index_name=index_name,
        output_field_mappings=[
            FieldMapping(source_field_name="/document/content_vector", target_field_name="content_vector"),
            FieldMapping(source_field_name="/document/title_vector", target_field_name="title_vector"),
            FieldMapping(source_field_name="/document/translatedText", target_field_name="translated")
        ]
    )
)

Creating skillset: nc-articles-nochunk-index
